In [1]:
# ===============================================
# 🧹 RELABEL & FILTER DATASET PLAIGIAT
# ===============================================
import pandas as pd

# 1️⃣ Load dataset dengan kolom similarity_score
df = pd.read_csv("dataset_pairs_clean.csv")

# Pastikan ada kolom 'similarity_score' — kalau belum, muat dari hasil sebelumnya
if 'similarity_score' not in df.columns:
    from sentence_transformers import SentenceTransformer, util
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    df['text_1'] = df['title_1'] + ". " + df['abstract_1']
    df['text_2'] = df['title_2'] + ". " + df['abstract_2']
    emb1 = model.encode(df['text_1'].tolist(), convert_to_tensor=True, show_progress_bar=True)
    emb2 = model.encode(df['text_2'].tolist(), convert_to_tensor=True, show_progress_bar=True)
    similarities = util.cos_sim(emb1, emb2)
    df['similarity_score'] = [float(similarities[i][i]) for i in range(len(df))]

# ===============================================
# 2️⃣ Relabel otomatis berdasarkan threshold
# ===============================================
threshold = 0.85
df['auto_label'] = (df['similarity_score'] >= threshold).astype(int)

# Tampilkan perbandingan label asli vs auto
print("\n📊 Perbandingan Label Asli vs Auto Relabel:")
print(pd.crosstab(df['label'], df['auto_label'], normalize='index').round(2))

# Simpan hasil relabel
df_relabel = df.copy()
df_relabel['label'] = df_relabel['auto_label']
df_relabel.to_csv("dataset_pairs_relabel.csv", index=False)
print(f"\n✅ Dataset hasil relabel otomatis disimpan ke dataset_pairs_relabel.csv")
print(f"Jumlah data: {len(df_relabel)}")

# ===============================================
# 3️⃣ Filter data ekstrem (clear separation)
# ===============================================
# Ambil hanya data yang sangat berbeda
plag = df_relabel[df_relabel['similarity_score'] >= 0.9]
not_plag = df_relabel[df_relabel['similarity_score'] <= 0.7]

# Samakan jumlah agar seimbang
sample_size = min(len(plag), len(not_plag))
filtered_df = pd.concat([
    plag.sample(sample_size, random_state=42),
    not_plag.sample(sample_size, random_state=42)
]).reset_index(drop=True)

# Simpan hasil
filtered_df.to_csv("dataset_pairs_filtered.csv", index=False)
print(f"\n✅ Dataset hasil filter ekstrem disimpan ke dataset_pairs_filtered.csv")
print(f"Jumlah data: {len(filtered_df)} (balanced {sample_size} vs {sample_size})")

# ===============================================
# 4️⃣ Statistik ringkas hasil filter
# ===============================================
print("\n📈 Statistik similarity setelah filter:")
print(filtered_df.groupby('label')['similarity_score'].describe())


e:\Flutter\saskia\check-plagiat\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\Flutter\saskia\check-plagiat\venv\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
e:\Flutter\saskia\check-plagiat\venv\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|██████████| 13/13 [00:32<00:00,  2.52s/it]



📊 Perbandingan Label Asli vs Auto Relabel:
auto_label     0     1
label                 
0           0.12  0.88
1           0.09  0.91

✅ Dataset hasil relabel otomatis disimpan ke dataset_pairs_relabel.csv
Jumlah data: 389

✅ Dataset hasil filter ekstrem disimpan ke dataset_pairs_filtered.csv
Jumlah data: 0 (balanced 0 vs 0)

📈 Statistik similarity setelah filter:
Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []
